# Домашняя работа: квантизация и дистилляция LLM

В этой работе вы самостоятельно проведёте эксперименты с квантизацией и дистилляцией LLM и проверите, сколько качества при этом теряет модель. Задача, на которую обучаемся, — разбор запросов к голосовому ассистенту из датасета MASSIVE (ru-RU): по запросу модель возвращает JSON с intent и slots. Подробнее: https://huggingface.co/datasets/AmazonScience/massive

### Сперва настроим окружение и подготовим код для экспериментов

In [1]:
# Раскомментируйте в новом окружении, затем перезапустите kernel:
# %pip install "torch==2.8.0" "transformers==4.55.2" "datasets==4.1.1" \
#     "accelerate==1.10.1" "llmcompressor==0.8.1" "compressed-tensors==0.12.2" \
#     "numpy==2.2.6" "pandas==2.2.3" "safetensors>=0.5" "huggingface-hub<1" tqdm

In [2]:
import os

os.environ["HF_HOME"] = "/home/edu/KAI/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/home/edu/KAI/.cache/huggingface/hub"
os.environ["HF_DATASETS_CACHE"] = "/home/edu/KAI/.cache/huggingface/datasets"

In [3]:
import gc
import json
import math
import random
import re
import shutil
import struct
from collections import Counter
from collections.abc import Sequence
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from importlib.metadata import version
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset, DatasetDict, load_dataset
from IPython.display import display
from tqdm.auto import tqdm
from transformers import (
    AutoConfig, AutoModelForCausalLM, AutoTokenizer,
    CompressedTensorsConfig, GenerationConfig, PreTrainedModel, PreTrainedTokenizerBase,
    Trainer, TrainingArguments, set_seed,
)

/home/edu/KAI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
SEED = 42
ARTIFACTS = Path("artifacts_compression_hw")
ARTIFACTS.mkdir(exist_ok=True)
set_seed(SEED)

def clean_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def ensure_empty_dir(path: str | Path) -> Path:
    path = Path(path)
    if path.exists() and any(path.iterdir()):
        raise FileExistsError(f"{path} уже существует и не пустой")
    path.mkdir(parents=True, exist_ok=True)
    return path

environment = {
    p: version(p) for p in (
        "torch", "transformers", "datasets", "llmcompressor",
        "compressed-tensors", "numpy", "safetensors",
    )
}
print(environment)
(ARTIFACTS / "environment.json").write_text(
    json.dumps(environment, indent=2), encoding="utf-8"
)

{'torch': '2.8.0', 'transformers': '4.55.2', 'datasets': '4.1.1', 'llmcompressor': '0.8.1', 'compressed-tensors': '0.12.2', 'numpy': '2.2.6', 'safetensors': '0.8.0'}


181

## Загрузка MASSIVE

Используем датасет MASSIVE. Все калибровочные примеры и данные для дистилляции берём только из train. Оцениваем на валидации (dev).

In [5]:
DATASET_ID = "AmazonScience/massive"
DATASET_REVISION = "ed58ac423a2f4121720918bf5301577edce4ffd3"  # refs/convert/parquet
LOCALE = "ru-RU"

dataset_url = f"hf://datasets/{DATASET_ID}@{DATASET_REVISION}/{LOCALE}"
data_files = {
    "train": f"{dataset_url}/train/0000.parquet",
    "validation": f"{dataset_url}/validation/0000.parquet",
    "test": f"{dataset_url}/test/0000.parquet",
}
raw = load_dataset("parquet", data_files=data_files)
print(raw)

Generating train split: 11514 examples [00:00, 457356.77 examples/s]
Generating validation split: 2033 examples [00:00, 311512.08 examples/s]
Generating test split: 2974 examples [00:00, 377766.81 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments'],
        num_rows: 11514
    })
    validation: Dataset({
        features: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments'],
        num_rows: 2033
    })
    test: Dataset({
        features: ['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments'],
        num_rows: 2974
    })
})


In [6]:
example_index = 0
example_raw = raw["train"][example_index]
print("Исходная запись MASSIVE:")
print(json.dumps(
    {key: example_raw[key] for key in ("id", "utt", "annot_utt", "intent")},
    ensure_ascii=False, indent=2,
))

Исходная запись MASSIVE:
{
  "id": "1",
  "utt": "разбуди меня в девять утра в пятницу",
  "annot_utt": "разбуди меня в [time : девять утра] в [date : пятницу]",
  "intent": 48
}


### Сконвертируем данные для калибровки

В MASSIVE текст запроса лежит в поле `utt`, intent записан числом, а slots отмечены прямо в тексте поля `annot_utt`, например `[time : семь утра]`. Модель должна по запросу вернуть JSON с названием intent и списком slots в виде пар «тип — значение». Поэтому преобразуем запись в `text` — исходный запрос — и `gold` — эталонный JSON-ответ. Такой формат удобнее для калибровки и подсчёта метрик.

In [7]:
SLOT_PATTERN = re.compile(r"\[([^:\[\]]+?)\s*:\s*([^\[\]]*?)\]")

def extract_slots(annot_utt: str) -> list[list[str]]:
    """Достаёт slots из разметки MASSIVE вида `[time : семь утра]`.

    Args:
        annot_utt: размеченный запрос из поля `annot_utt`.

    Returns:
        Список пар `[тип, значение]` в порядке появления в запросе.
    """
    return [[m.group(1).strip(), m.group(2).strip()]
            for m in SLOT_PATTERN.finditer(annot_utt)]

def target_json(intent: str, slots: list[list[str]]) -> str:
    """Собирает эталонный ответ в компактный JSON.

    Args:
        intent: название intent.
        slots: список пар `[тип, значение]`.

    Returns:
        Строка `{"intent":...,"slots":[...]}` без пробелов между полями.
    """
    return json.dumps(
        {"intent": intent, "slots": slots},
        ensure_ascii=False, separators=(",", ":"),
    )

def prepare_massive(raw_splits: DatasetDict) -> DatasetDict:
    """Переводит сплиты MASSIVE в формат запрос - эталонный JSON.

    Args:
        raw_splits: исходные сплиты MASSIVE с полями `id`, `utt`, `intent`, `annot_utt`.

    Returns:
        Сплиты с полями `id`, `text` (запрос) и `gold` (эталонный JSON).
    """
    intent_feature = raw_splits["train"].features["intent"]

    def convert(row: dict[str, Any]) -> dict[str, str]:
        intent = intent_feature.int2str(row["intent"])
        slots = extract_slots(row["annot_utt"])
        return {
            "id": str(row["id"]), "text": row["utt"],
            "gold": target_json(intent, slots),
        }

    return DatasetDict({
        split: ds.map(convert, remove_columns=ds.column_names)
        for split, ds in raw_splits.items()
    })

In [8]:

data = prepare_massive(raw)
INTENTS = list(raw["train"].features["intent"].names)
SLOT_TYPES = sorted({
    kind for row in raw["train"]
    for kind, value in extract_slots(row["annot_utt"])
})
train_data = data["train"]
import os
DEV_SAMPLES = int(os.getenv("DEV_SAMPLES", "256"))
dev_data = data["validation"]
if DEV_SAMPLES is not None:
    dev_data = dev_data.shuffle(seed=SEED).select(range(min(DEV_SAMPLES, len(dev_data))))
print("Число классов:", len(INTENTS), "типов slots:", len(SLOT_TYPES))

Map: 100%|██████████| 2974/2974 [00:00<00:00, 12363.23 examples/s]


Число классов: 60 типов slots: 54


In [9]:
example_prepared = train_data[example_index]
print("Запрос модели (text):")
print(example_prepared["text"])
print("\nЭталонный ответ (gold):")
print(json.dumps(json.loads(example_prepared["gold"]), ensure_ascii=False, indent=2))

Запрос модели (text):
разбуди меня в девять утра в пятницу

Эталонный ответ (gold):
{
  "intent": "alarm_set",
  "slots": [
    [
      "time",
      "девять утра"
    ],
    [
      "date",
      "пятницу"
    ]
  ]
}


### Генерация ответов
Всем моделям даём одинаковую инструкцию и одинаковые описания допустимых классов. Генерация детерминирована: greedy decoding, одинаковый лимит токенов.

In [10]:
import os
BASE_MODEL_ID = os.getenv("BASE_MODEL_ID", "Qwen/Qwen3-0.6B") # можно пробовать модели того же семейства, но поменьше, чтобы считалось быстрее

# Значения используются как пример, при необходимости можно регулировать
MAX_PROMPT_TOKENS = 2048
MAX_NEW_TOKENS = 128
GENERATE_BATCH_SIZE = int(os.getenv("BATCH_SIZE", "8"))

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda:0


In [11]:
system_prompt = (
    'Определи intent и извлеки slots. Верни только JSON вида '
    '{"intent":"...","slots":[["тип","значение"],...]}. '
    'Значения slots копируй из запроса. Если slots нет, верни пустой список. '
    "Не добавляй объяснения.\nДопустимые intents: " + ", ".join(INTENTS)
    + "\nДопустимые типы slots: " + ", ".join(SLOT_TYPES)
)

def encode_prompt(tokenizer: PreTrainedTokenizerBase, text: str) -> list[int]:
    """Токенизирует запрос в chat-формате с общим system prompt.

    Args:
        tokenizer: токенизатор модели с chat template.
        text: запрос пользователя.

    Returns:
        Token ids промпта, заканчивающиеся началом ответа ассистента.
    """
    ids = tokenizer.apply_chat_template(
        [{"role": "system", "content": system_prompt},
         {"role": "user", "content": text}],
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=False,
    )
    if len(ids) > MAX_PROMPT_TOKENS:
        raise ValueError(f"Prompt имеет {len(ids)} токенов. Увеличьте лимит, не обрезайте запрос.")
    return ids

def generation_config(tokenizer: PreTrainedTokenizerBase) -> GenerationConfig:
    pad_token_id = tokenizer.pad_token_id
    if pad_token_id is None:
        pad_token_id = tokenizer.eos_token_id
    return GenerationConfig(
        do_sample=False, num_beams=1, max_new_tokens=MAX_NEW_TOKENS,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=pad_token_id,
        use_cache=True,
    )

def trim_answer(token_ids: list[int], eos_token_id: int, pad_token_id: int) -> list[int]:
    """Отрезает паддинг, после генерации

    Returns:
        Токены ответа; eos остаётся последним, если модель его сгенерировала.
    """
    result = []
    for token in token_ids:
        if token == eos_token_id:
            result.append(token)
            break
        if token == pad_token_id:
            break
        result.append(token)
    return result

@torch.inference_mode()
def generate_answer_ids(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    prompts: Sequence[list[int]],
) -> list[list[int]]:
    """Жадно генерирует ответы на батч промптов.

    Args:
        model: causal LM.
        tokenizer: токенизатор модели.
        prompts: token ids промптов; паддинг добавляется слева.

    Returns:
        Токены ответа для каждого промпта в том же порядке, без промпта и паддинга.
    """
    config = generation_config(tokenizer)
    width = max(len(ids) for ids in prompts)
    input_ids, attention_mask = [], []

    for ids in prompts:
        pad = width - len(ids)
        input_ids.append([config.pad_token_id] * pad + ids)
        attention_mask.append([0] * pad + [1] * len(ids))
    input_ids = torch.tensor(input_ids, device=model.device)
    attention_mask = torch.tensor(attention_mask, device=model.device)

    output = model.generate(
        input_ids=input_ids, attention_mask=attention_mask,
        generation_config=config,
    )
    return [
        trim_answer(row, config.eos_token_id, config.pad_token_id)
        for row in output[:, input_ids.shape[1]:].tolist()
    ]

In [12]:
BASELINE_PATH = ARTIFACTS / "baseline"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
baseline_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, torch_dtype=torch.bfloat16,
    device_map=DEVICE,
).eval()
ensure_empty_dir(BASELINE_PATH)
baseline_model.save_pretrained(BASELINE_PATH, safe_serialization=True)
tokenizer.save_pretrained(BASELINE_PATH)
print("Исходная модель:", BASE_MODEL_ID)
print("Пример prompt, токенов:", len(encode_prompt(tokenizer, train_data[0]["text"])))

Исходная модель: Qwen/Qwen3-0.6B
Пример prompt, токенов: 504


## Метрика качества

$$
Q = \tfrac{1}{2} \operatorname{IntentAccuracy} + \tfrac{1}{2} \operatorname{SlotMicroF1}.
$$

Slots сравниваются как **мультимножество** пар `(тип, значение)`: порядок не влияет, лишние дубликаты дают false positives. Неверный JSON или нарушение схемы считается ошибкой всего ответа.

Дополнительно считаем `frame_exact_match` и `valid_json_rate`. Если во всей выборке нет ни одного gold/predicted slot, slot F1 равен 1 только при валидности всех ответов, иначе 0.

In [13]:
def normalize_value(value: str) -> str:
    """Приводит значение slot к виду для сравнения."""
    return " ".join(value.casefold().split())

def parse_answer(
    text: str, intents: Sequence[str], slot_types: Sequence[str],
) -> dict[str, Any] | None:
    """Разбирает ответ модели и проверяет формат.

    Args:
        text: сырой ответ модели.
        intents: допустимые названия intent.
        slot_types: допустимые типы slots.

    Returns:
        Словарь с ключами `intent` и `slots` или None, если ответ невалиден.
    """
    try:
        obj = json.loads(text.strip())
        if not isinstance(obj, dict) or set(obj) != {"intent", "slots"}:
            return None
        if not isinstance(obj["intent"], str) or obj["intent"] not in intents:
            return None
        if not isinstance(obj["slots"], list):
            return None
        for slot in obj["slots"]:
            if not isinstance(slot, list) or len(slot) != 2:
                return None
            kind, value = slot
            if (not isinstance(kind, str) or kind not in slot_types
                    or not isinstance(value, str) or not value.strip()):
                return None
        return obj
    except (ValueError, TypeError):
        return None

def slot_counter(obj: dict[str, Any]) -> Counter:
    """Превращает slots ответа в мультимножество

    Args:
        obj: разобранный ответ с ключом `slots`.

    Returns:
        Counter пар `(тип, нормализованное значение)`.
    """
    return Counter((kind, normalize_value(value)) for kind, value in obj["slots"])

def compute_metrics(
    predictions: Sequence[str],
    golds: Sequence[str],
    intents: Sequence[str],
    slot_types: Sequence[str],
) -> dict[str, float]:
    """Считает метрики качества по ответам модели.

    Args:
        predictions: ответы модели.
        golds: эталонные JSON в том же порядке.
        intents: допустимые названия intent.
        slot_types: допустимые типы slots.

    Returns:
        Словарь с `quality`, `intent_accuracy`, `slot_micro_f1`,
        `frame_exact_match`, `valid_json_rate` и числом примеров `n`.
    """
    if len(predictions) != len(golds) or not golds:
        raise ValueError("Нужны непустые списки одинаковой длины.")
    correct_intent = exact = valid = tp = fp = fn = 0
    for text, gold_text in zip(predictions, golds):
        gold = parse_answer(gold_text, intents, slot_types)
        if gold is None:
            raise ValueError("Невалидная gold-разметка")
        pred = parse_answer(text, intents, slot_types)
        gold_slots = slot_counter(gold)
        pred_slots = Counter() if pred is None else slot_counter(pred)
        tp += sum((pred_slots & gold_slots).values())
        fp += sum((pred_slots - gold_slots).values())
        fn += sum((gold_slots - pred_slots).values())
        if pred is not None:
            valid += 1
            intent_ok = pred["intent"] == gold["intent"]
            correct_intent += int(intent_ok)
            exact += int(intent_ok and pred_slots == gold_slots)
    n = len(golds)
    denom = 2 * tp + fp + fn
    slot_f1 = 2 * tp / denom if denom else float(valid == n)
    intent_acc = correct_intent / n
    return {
        "quality": 0.5 * (intent_acc + slot_f1),
        "intent_accuracy": intent_acc,
        "slot_micro_f1": slot_f1,
        "frame_exact_match": exact / n,
        "valid_json_rate": valid / n,
        "n": n,
    }

def evaluate(
    model: PreTrainedModel, tokenizer: PreTrainedTokenizerBase, ds: Dataset,
) -> tuple[dict[str, float], list[str]]:
    """Генерирует ответы на выборку батчами и считает метрики.

    Args:
        model: оцениваемая модель.
        tokenizer: её токенизатор.
        ds: выборка с полями `text` и `gold`.

    Returns:
        Метрики из compute_metrics и список ответов модели в порядке выборки.
    """
    model.eval()
    texts = ds["text"]
    predictions = []

    for start in tqdm(range(0, len(texts), GENERATE_BATCH_SIZE), desc="dev"):
        prompts = [encode_prompt(tokenizer, t) for t in texts[start:start+GENERATE_BATCH_SIZE]]
        answers = generate_answer_ids(model, tokenizer, prompts)
        predictions.extend(tokenizer.batch_decode(answers, skip_special_tokens=True))

    metrics = compute_metrics(predictions, ds["gold"], INTENTS, SLOT_TYPES)
    return metrics, predictions

def evaluate_checkpoint(
    model_path: str | Path, ds: Dataset,
) -> tuple[dict[str, float], list[str]]:
    model_path = str(model_path)
    config = AutoConfig.from_pretrained(model_path)
    quantization_kwargs = {}

    if getattr(config, "quantization_config", None):
        quantization_kwargs["quantization_config"] = CompressedTensorsConfig(
            run_compressed=True
        )

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        device_map=DEVICE,
        **quantization_kwargs,
    )

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        return evaluate(model, tokenizer, ds)
    finally:
        del model
        clean_memory()

Посчитаем метрики бейзлайна.

In [ ]:
baseline_metrics, baseline_predictions = evaluate(baseline_model, tokenizer, dev_data)
print(baseline_metrics)
(ARTIFACTS / "baseline_dev.json").write_text(
    json.dumps(baseline_metrics, indent=2), encoding="utf-8"
)
results = [{
    "run": "baseline", "path": str(BASELINE_PATH),
    "method": "dense", "calibration": None, **baseline_metrics,
}]
del baseline_model
clean_memory()

def save_eval(name, metrics, predictions, **params):
    payload = {"run": name, **params, **metrics}
    (ARTIFACTS / f"{name}_dev.json").write_text(json.dumps(payload, indent=2))
    with (ARTIFACTS / f"{name}_predictions.jsonl").open("w", encoding="utf-8") as f:
        for row, pred in zip(dev_data, predictions):
            f.write(json.dumps({**row, "prediction": pred}, ensure_ascii=False) + "\n")
    return payload
save_eval("baseline", baseline_metrics, baseline_predictions, model=BASE_MODEL_ID)


dev: 100%|██████████| 32/32 [00:54<00:00,  1.69s/it]


{'quality': 0.08453050947867298, 'intent_accuracy': 0.140625, 'slot_micro_f1': 0.02843601895734597, 'frame_exact_match': 0.0078125, 'valid_json_rate': 0.421875, 'n': 256}


{'run': 'baseline',
 'model': 'Qwen/Qwen3-0.6B',
 'quality': 0.08453050947867298,
 'intent_accuracy': 0.140625,
 'slot_micro_f1': 0.02843601895734597,
 'frame_exact_match': 0.0078125,
 'valid_json_rate': 0.421875,
 'n': 256}

: 

## Квантизация (5 баллов)

Используя код с семинара выполните следующие задания:
1) На примере одного из методов откалибруйте бейзлайн на MASSIVE train и на WikiText, сравните качество на dev. Есть ли влияние калибровочного датасета на метрику? (1 балл)

2) Поэкспериментируйте с разными конфигурациями методов AWQ, GPTQ, SmoothQuant (3 балла)

    Запустите минимум три конфигурации, пробуйте менять разные факторы:

    - битность
    - число калибровочных примеров;
    - group size
    - symmetric / asymmetric quantization;
    - smoothing_strength
    - Дополнительно (+1 балл) — какие слои оставить в исходной точности

3) Объясните результат — какие параметры наиболее значимы? (1 балл)


In [ ]:
from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier
from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor.modifiers.smoothquant import SmoothQuantModifier

wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
wiki = wiki.filter(lambda row: len(row["text"].split()) >= 20).shuffle(seed=SEED)
cal_train = train_data.shuffle(seed=SEED)
configs = [
    ("gptq_massive", "GPTQ", "massive", "W4A16", 128, None),
    ("gptq_wiki", "GPTQ", "wiki", "W4A16", 128, None),
    ("awq_massive", "AWQ", "massive", "W4A16", 128, None),
    ("smooth_massive", "SmoothQuant", "massive", "W8A8", 128, 0.8),
    ("gptq_64", "GPTQ", "massive", "W4A16", 64, None),
    ("smooth_05", "SmoothQuant", "massive", "W8A8", 128, 0.5),
]
for name, method, calibration, scheme, n, strength in configs:
    path = ARTIFACTS / name
    if calibration == "massive":
        texts = [tokenizer.decode(encode_prompt(tokenizer, row["text"]), skip_special_tokens=False) for row in cal_train.select(range(n))]
    else:
        texts = wiki.select(range(n))["text"]
    ds = Dataset.from_dict({"text": texts})
    kwargs = dict(targets="Linear", scheme=scheme, ignore=["lm_head"])
    if method == "AWQ":
        recipe = AWQModifier(**kwargs)
    elif method == "GPTQ":
        recipe = GPTQModifier(**kwargs)
    else:
        recipe = [SmoothQuantModifier(smoothing_strength=strength), GPTQModifier(**kwargs)]
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, torch_dtype=torch.bfloat16)
    oneshot(model=model, dataset=ds, recipe=recipe, max_seq_length=MAX_PROMPT_TOKENS,
            num_calibration_samples=n, output_dir=str(path))
    tokenizer.save_pretrained(path)
    del model
    clean_memory()
    metrics, preds = evaluate_checkpoint(path, dev_data)
    results.append(save_eval(name, metrics, preds, method=method, calibration=calibration,
                             scheme=scheme, calibration_samples=n, smoothing_strength=strength))
    pd.DataFrame(results).to_csv(ARTIFACTS / "quantization_results.csv", index=False)
display(pd.DataFrame(results))


Tokenizing: 100%|██████████| 128/128 [00:00<00:00, 651.00 examples/s]


2026-11-26T00:40:26.933019+0300 | reset | INFO - Compression lifecycle reset
2026-11-26T00:40:26.939791+0300 | _create_default_logger | INFO - Logging all LLM Compressor modifier-level logs to sparse_logs/26-11-2026_00.40.26.log
2026-11-26T00:40:26.940603+0300 | from_modifiers | INFO - Creating recipe from modifiers
2026-11-26T00:40:26.968349+0300 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-11-26T00:40:26.969030+0300 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

2026-11-26T00:40:29.591338+0300 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 128 samples


2026-11-26T00:40:30.092618+0300 | compress | METRIC - time 0.50s
2026-11-26T00:40:30.093463+0300 | compress | METRIC - error 72.01
2026-11-26T00:40:30.094424+0300 | compress | METRIC - GPU 0 | usage: 3.01% | total memory: 85 GB
2026-11-26T00:40:30.094850+0300 | compress | METRIC - GPU 1 | usage: 4.12% | total memory: 85 GB
2026-11-26T00:40:30.095381+0300 | compress | METRIC - Compressed module size: 4.243456 MB
2026-11-26T00:40:30.095965+0300 | compress_modules | INFO - Quantizing model.layers.0.self_attn.k_proj using 128 samples
2026-11-26T00:40:30.531947+0300 | compress | METRIC - time 0.44s
2026-11-26T00:40:30.532891+0300 | compress | METRIC - error 31.79
2026-11-26T00:40:30.533546+0300 | compress | METRIC - GPU 0 | usage: 3.01% | total memory: 85 GB
2026-11-26T00:40:30.533928+0300 | compress | METRIC - GPU 1 | usage: 4.12% | total memory: 85 GB
2026-11-26T00:40:30.534450+0300 | compress | METRIC - Compressed module size: 2.121728 MB
2026-11-26T00:40:30.535015+0300 | compress_module

(2/29): Calibrating: 100%|██████████| 128/128 [00:01<00:00, 107.18it/s]

... (171 outputs left out)

2026-11-26T00:51:38.088690+0300 | reset | INFO - Compression lifecycle reset
2026-11-26T00:51:38.091656+0300 | from_modifiers | INFO - Creating recipe from modifiers
2026-11-26T00:51:38.116770+0300 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...



Resolving mapping 1/4 (0 skipped): : 28it [00:00, 1901.41it/s]
Resolving mapping 2/4 (27 skipped): : 28it [00:00, 2732.89it/s]
Resolving mapping 3/4 (0 skipped): : 28it [00:00, 1658.60it/s]
Resolving mapping 4/4 (0 skipped): : 28it [00:00, 2689.46it/s]

2026-11-26T00:51:38.175557+0300 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-11-26T00:51:38.176081+0300 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `AWQModifier`



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The

In [1]:
import os

os.environ["HF_HOME"] = "/home/edu/KAI/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/home/edu/KAI/.cache/huggingface/hub"
os.environ["HF_DATASETS_CACHE"] = "/home/edu/KAI/.cache/huggingface/datasets"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import json
import math
import random
import re
import shutil
import struct

from pathlib import Path
from collections import Counter
from collections.abc import Sequence
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from importlib.metadata import version
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from datasets import Dataset, DatasetDict, load_dataset
from IPython.display import display
from tqdm.auto import tqdm

from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    CompressedTensorsConfig,
    GenerationConfig,
    PreTrainedModel,
    PreTrainedTokenizerBase,
    Trainer,
    TrainingArguments,
    set_seed,
)

from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier
from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor.modifiers.smoothquant import SmoothQuantModifier

SEED = 42
ARTIFACTS = Path("artifacts_compression_hw")
BASELINE_PATH = ARTIFACTS / "baseline"
BASE_MODEL_ID = os.getenv("BASE_MODEL_ID", "Qwen/Qwen3-0.6B")
MAX_PROMPT_TOKENS = 2048
MAX_NEW_TOKENS = 128
GENERATE_BATCH_SIZE = int(os.getenv("BATCH_SIZE", "8"))
DEV_SAMPLES = int(os.getenv("DEV_SAMPLES", "256"))
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

set_seed(SEED)


def clean_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def ensure_empty_dir(path):
    path = Path(path)
    if path.exists() and any(path.iterdir()):
        raise FileExistsError(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


DATASET_ID = "AmazonScience/massive"
DATASET_REVISION = "ed58ac423a2f4121720918bf5301577edce4ffd3"
LOCALE = "ru-RU"

dataset_url = f"hf://datasets/{DATASET_ID}@{DATASET_REVISION}/{LOCALE}"

raw = load_dataset(
    "parquet",
    data_files={
        split: f"{dataset_url}/{split}/0000.parquet"
        for split in ["train", "validation", "test"]
    },
)

SLOT_PATTERN = re.compile(r"\[([^:\[\]]+?)\s*:\s*([^\[\]]*?)\]")


def extract_slots(annot_utt):
    return [
        [m.group(1).strip(), m.group(2).strip()]
        for m in SLOT_PATTERN.finditer(annot_utt)
    ]


def target_json(intent, slots):
    return json.dumps(
        {"intent": intent, "slots": slots},
        ensure_ascii=False,
        separators=(",", ":"),
    )


def prepare_massive(raw_splits):
    intent_feature = raw_splits["train"].features["intent"]

    def convert(row):
        return {
            "id": str(row["id"]),
            "text": row["utt"],
            "gold": target_json(
                intent_feature.int2str(row["intent"]),
                extract_slots(row["annot_utt"]),
            ),
        }

    return DatasetDict({
        split: ds.map(convert, remove_columns=ds.column_names)
        for split, ds in raw_splits.items()
    })


data = prepare_massive(raw)
INTENTS = list(raw["train"].features["intent"].names)

SLOT_TYPES = sorted({
    kind
    for row in raw["train"]
    for kind, value in extract_slots(row["annot_utt"])
})

train_data = data["train"]

dev_data = data["validation"].shuffle(seed=SEED).select(
    range(min(DEV_SAMPLES, len(data["validation"])))
)

system_prompt = (
    'Определи intent и извлеки slots. Верни только JSON вида '
    '{"intent":"...","slots":[["тип","значение"],...]}. '
    'Значения slots копируй из запроса. Если slots нет, верни пустой список. '
    "Не добавляй объяснения.\nДопустимые intents: "
    + ", ".join(INTENTS)
    + "\nДопустимые типы slots: "
    + ", ".join(SLOT_TYPES)
)


def encode_prompt(tokenizer, text):
    ids = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text},
        ],
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=False,
    )

    if len(ids) > MAX_PROMPT_TOKENS:
        raise ValueError(f"Prompt имеет {len(ids)} токенов")

    return ids


def generation_config(tokenizer):
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id

    return GenerationConfig(
        do_sample=False,
        num_beams=1,
        max_new_tokens=MAX_NEW_TOKENS,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=pad_id,
        use_cache=True,
    )


def trim_answer(token_ids, eos_token_id, pad_token_id):
    result = []

    for token in token_ids:
        if token == eos_token_id:
            result.append(token)
            break
        if token == pad_token_id:
            break
        result.append(token)

    return result


@torch.inference_mode()
def generate_answer_ids(model, tokenizer, prompts):
    config = generation_config(tokenizer)
    width = max(len(ids) for ids in prompts)

    input_ids, attention_mask = [], []

    for ids in prompts:
        pad = width - len(ids)
        input_ids.append([config.pad_token_id] * pad + ids)
        attention_mask.append([0] * pad + [1] * len(ids))

    input_ids = torch.tensor(input_ids, device=model.device)
    attention_mask = torch.tensor(attention_mask, device=model.device)

    output = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        generation_config=config,
    )

    return [
        trim_answer(row, config.eos_token_id, config.pad_token_id)
        for row in output[:, input_ids.shape[1]:].tolist()
    ]


def normalize_value(value):
    return " ".join(value.casefold().split())


def parse_answer(text, intents, slot_types):
    try:
        obj = json.loads(text.strip())

        if not isinstance(obj, dict) or set(obj) != {"intent", "slots"}:
            return None

        if not isinstance(obj["intent"], str) or obj["intent"] not in intents:
            return None

        if not isinstance(obj["slots"], list):
            return None

        for slot in obj["slots"]:
            if not isinstance(slot, list) or len(slot) != 2:
                return None

            kind, value = slot

            if (
                not isinstance(kind, str)
                or kind not in slot_types
                or not isinstance(value, str)
                or not value.strip()
            ):
                return None

        return obj

    except (ValueError, TypeError):
        return None


def slot_counter(obj):
    return Counter(
        (kind, normalize_value(value))
        for kind, value in obj["slots"]
    )


def compute_metrics(predictions, golds, intents, slot_types):
    if len(predictions) != len(golds) or not golds:
        raise ValueError("Invalid predictions")

    correct_intent = exact = valid = tp = fp = fn = 0

    for text, gold_text in zip(predictions, golds):
        gold = parse_answer(gold_text, intents, slot_types)
        pred = parse_answer(text, intents, slot_types)

        if gold is None:
            raise ValueError("Невалидная gold-разметка")

        gold_slots = slot_counter(gold)
        pred_slots = Counter() if pred is None else slot_counter(pred)

        tp += sum((pred_slots & gold_slots).values())
        fp += sum((pred_slots - gold_slots).values())
        fn += sum((gold_slots - pred_slots).values())

        if pred is not None:
            valid += 1
            intent_ok = pred["intent"] == gold["intent"]
            correct_intent += int(intent_ok)
            exact += int(intent_ok and pred_slots == gold_slots)

    n = len(golds)
    denom = 2 * tp + fp + fn
    slot_f1 = 2 * tp / denom if denom else float(valid == n)
    intent_acc = correct_intent / n

    return {
        "quality": 0.5 * (intent_acc + slot_f1),
        "intent_accuracy": intent_acc,
        "slot_micro_f1": slot_f1,
        "frame_exact_match": exact / n,
        "valid_json_rate": valid / n,
        "n": n,
    }


def evaluate(model, tokenizer, ds):
    model.eval()
    texts = ds["text"]
    predictions = []

    for start in tqdm(
        range(0, len(texts), GENERATE_BATCH_SIZE),
        desc="dev",
    ):
        prompts = [
            encode_prompt(tokenizer, t)
            for t in texts[start:start + GENERATE_BATCH_SIZE]
        ]

        answers = generate_answer_ids(model, tokenizer, prompts)

        predictions.extend(
            tokenizer.batch_decode(answers, skip_special_tokens=True)
        )

    return compute_metrics(
        predictions, ds["gold"], INTENTS, SLOT_TYPES
    ), predictions


def evaluate_checkpoint(model_path, ds):
    model_path = str(model_path)
    config = AutoConfig.from_pretrained(model_path)
    quantization_kwargs = {}

    if getattr(config, "quantization_config", None):
        quantization_kwargs["quantization_config"] = (
            CompressedTensorsConfig(run_compressed=True)
        )

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        device_map=DEVICE,
        **quantization_kwargs,
    )

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        return evaluate(model, tokenizer, ds)
    finally:
        del model
        clean_memory()


def save_eval(name, metrics, predictions, **params):
    payload = {"run": name, **params, **metrics}

    with (ARTIFACTS / f"{name}_predictions.jsonl").open(
        "w", encoding="utf-8"
    ) as f:
        for row, pred in zip(dev_data, predictions):
            f.write(
                json.dumps(
                    {**row, "prediction": pred},
                    ensure_ascii=False,
                ) + "\n"
            )

    (ARTIFACTS / f"{name}_dev.json").write_text(
        json.dumps(payload, indent=2),
        encoding="utf-8",
    )

    return payload


tokenizer = AutoTokenizer.from_pretrained(BASELINE_PATH)

baseline_metrics = json.loads(
    (ARTIFACTS / "baseline_dev.json").read_text()
)

baseline_metrics = {
    k: baseline_metrics[k]
    for k in [
        "quality",
        "intent_accuracy",
        "slot_micro_f1",
        "frame_exact_match",
        "valid_json_rate",
        "n",
    ]
}

baseline_predictions = [
    json.loads(line)["prediction"]
    for line in (ARTIFACTS / "baseline_predictions.jsonl").read_text().splitlines()
]

results = [{
    "run": "baseline",
    "path": str(BASELINE_PATH),
    "method": "dense",
    "calibration": None,
    **baseline_metrics,
}]

assert baseline_metrics["n"] == len(dev_data)

wiki = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="train",
)

wiki = wiki.filter(
    lambda row: len(row["text"].split()) >= 20
).shuffle(seed=SEED)

cal_train = train_data.shuffle(seed=SEED)

configs = [
    ("gptq_massive", "GPTQ", "massive", "W4A16", 128, None),
    ("gptq_wiki", "GPTQ", "wiki", "W4A16", 128, None),
    ("awq_massive", "AWQ", "massive", "W4A16", 128, None),
    ("smooth_massive", "SmoothQuant", "massive", "W8A8", 128, 0.8),
    ("gptq_64", "GPTQ", "massive", "W4A16", 64, None),
    ("smooth_05", "SmoothQuant", "massive", "W8A8", 128, 0.5),
]

csv_path = ARTIFACTS / "quantization_results.csv"

for i, (name, method, calibration, scheme, n, strength) in enumerate(configs, 1):
    path = ARTIFACTS / name
    metrics_path = ARTIFACTS / f"{name}_dev.json"
    predictions_path = ARTIFACTS / f"{name}_predictions.jsonl"

    print(f"\n[{i}/{len(configs)}] {name}", flush=True)

    if metrics_path.exists() and predictions_path.exists():
        record = json.loads(metrics_path.read_text())
        assert record["n"] == len(dev_data)

        results.append(record)
        pd.DataFrame(results).to_csv(csv_path, index=False)

        print("Restored completed experiment", flush=True)
        print(record, flush=True)
        continue

    checkpoint_exists = (
        (path / "config.json").exists()
        and (path / "tokenizer_config.json").exists()
        and bool(list(path.glob("*.safetensors")))
    )

    if not checkpoint_exists:
        if calibration == "massive":
            texts = [
                tokenizer.decode(
                    encode_prompt(tokenizer, row["text"]),
                    skip_special_tokens=False,
                )
                for row in cal_train.select(range(n))
            ]
        else:
            texts = wiki.select(range(n))["text"]

        ds = Dataset.from_dict({"text": texts})

        kwargs = dict(
            targets="Linear",
            scheme=scheme,
            ignore=["lm_head"],
        )

        if method == "AWQ":
            recipe = AWQModifier(**kwargs)
        elif method == "GPTQ":
            recipe = GPTQModifier(**kwargs)
        else:
            recipe = [
                SmoothQuantModifier(smoothing_strength=strength),
                GPTQModifier(**kwargs),
            ]

        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            torch_dtype=torch.bfloat16,
        )

        oneshot(
            model=model,
            dataset=ds,
            recipe=recipe,
            max_seq_length=MAX_PROMPT_TOKENS,
            num_calibration_samples=n,
            output_dir=str(path),
        )

        tokenizer.save_pretrained(path)

        del model, recipe, ds, texts
        clean_memory()

    else:
        print("Restored checkpoint", flush=True)

    metrics, preds = evaluate_checkpoint(path, dev_data)

    results.append(save_eval(
        name,
        metrics,
        preds,
        method=method,
        calibration=calibration,
        scheme=scheme,
        calibration_samples=n,
        smoothing_strength=strength,
    ))

    pd.DataFrame(results).to_csv(csv_path, index=False)
    print(metrics, flush=True)

display(pd.DataFrame(results))

/home/edu/KAI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



[1/6] gptq_massive
Restored completed experiment
{'run': 'gptq_massive', 'method': 'GPTQ', 'calibration': 'massive', 'scheme': 'W4A16', 'calibration_samples': 128, 'smoothing_strength': None, 'quality': 0.017578125, 'intent_accuracy': 0.03515625, 'slot_micro_f1': 0.0, 'frame_exact_match': 0.0078125, 'valid_json_rate': 0.0703125, 'n': 256}

[2/6] gptq_wiki
Restored completed experiment
{'run': 'gptq_wiki', 'method': 'GPTQ', 'calibration': 'wiki', 'scheme': 'W4A16', 'calibration_samples': 128, 'smoothing_strength': None, 'quality': 0.0, 'intent_accuracy': 0.0, 'slot_micro_f1': 0.0, 'frame_exact_match': 0.0, 'valid_json_rate': 0.01953125, 'n': 256}

[3/6] awq_massive
Restored completed experiment
{'run': 'awq_massive', 'method': 'AWQ', 'calibration': 'massive', 'scheme': 'W4A16', 'calibration_samples': 128, 'smoothing_strength': None, 'quality': 0.001953125, 'intent_accuracy': 0.00390625, 'slot_micro_f1': 0.0, 'frame_exact_match': 0.0, 'valid_json_rate': 0.0234375, 'n': 256}

[4/6] smoot

,run,path,method,calibration,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n,scheme,calibration_samples,smoothing_strength
0,baseline,artifacts_compression_hw/baseline,dense,None,0.084531,0.140625,0.028436,0.007812,0.421875,256,NaN,NaN,NaN
1,gptq_massive,NaN,GPTQ,massive,0.017578,0.035156,0.000000,0.007812,0.070312,256,W4A16,128.0,NaN
2,gptq_wiki,NaN,GPTQ,wiki,0.000000,0.000000,0.000000,0.000000,0.019531,256,W4A16,128.0,NaN
3,awq_massive,NaN,AWQ,massive,0.001953,0.003906,0.000000,0.000000,0.023438,256,W4A16,128.0,NaN
4,smooth_massive,NaN,SmoothQuant,massive,0.074548,0.125000,0.024096,0.003906,0.453125,256,W8A8,128.0,0.8
5,gptq_64,NaN,GPTQ,massive,0.005859,0.011719,0.000000,0.000000,0.046875,256,W4A16,64.0,NaN
6,smooth_05,NaN,SmoothQuant,massive,0.073714,0.113281,0.034146,0.011719,0.394531,256,W8A8,128.0,0.5


### Выводы
Заполняются по фактическим результатам запуска (summary.json).

По результатам экспериментов калибровка на MASSIVE оказалась эффективнее калибровки на WikiText: качество GPTQ составило 0.0176 против 0. Это показывает важность выбора калибровочного датасета, соответствующего целевой задаче.

Увеличение количества калибровочных примеров с 64 до 128 повысило качество GPTQ с 0.0059 до 0.0176.

Лучший результат среди квантизированных моделей показал SmoothQuant (W8A8) с smoothing_strength = 0.8: качество составило 0.0745 при качестве бейзлайна 0.0845. Таким образом, удалось сохранить около 88% исходного качества.

Наибольшие различия наблюдаются между конфигурациями W4A16 и W8A8. Изменение smoothing_strength с 0.5 до 0.8 оказало значительно меньшее влияние. При этом отдельно оценить влияние битности нельзя, поскольку одновременно изменялся метод квантизации.

## Hard-label дистилляция (5 баллов)

Теперь реализуем hard-label дистилляцию знаний от учителя **Qwen3-32B** в бейзлайн.

### Задания
1) Измерьте качество учителя на dev и сравните с бейзлайном: насколько учитель сильнее? (1 балл)
2) Получите жадные ответы учителя на сэмпле из train. Используйте батч для ускорения при необходимости (1 балл)
3) Реализуйте код для обучения ученика на ответах учителя (1 балл)
4) Проведите эксперименты с обучением, получите модель ученика, отклоняющуюся на dev не более чем на 10% от метрики учителя (2 балла)


### Измерим качество учителя и сравним с бейзлайном (1 балл)

In [2]:
STUDENT_MODEL_ID = BASE_MODEL_ID
TEACHER_MODEL_ID = "Qwen/Qwen3-32B"
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID)


In [3]:
teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto",
).eval()


Loading checkpoint shards: 100%|██████████| 17/17 [00:27<00:00,  1.63s/it]


In [4]:
teacher_metrics, teacher_predictions = evaluate(teacher, teacher_tokenizer, dev_data)
save_eval("teacher", teacher_metrics, teacher_predictions, model=TEACHER_MODEL_ID)
print(teacher_metrics)


dev: 100%|██████████| 32/32 [02:20<00:00,  4.40s/it]

{'quality': 0.5856026785714286, 'intent_accuracy': 0.7890625, 'slot_micro_f1': 0.3821428571428571, 'frame_exact_match': 0.3203125, 'valid_json_rate': 0.98046875, 'n': 256}


### Получим ответы учителя на трейне (1 балл)

Не забывайте выгружать модели и очищать память.

In [5]:
DISTILL_SAMPLES = int(os.getenv("DISTILL_SAMPLES", "1024"))

def collect_teacher_answers(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    rows: Dataset,
    output_path: str | Path,
) -> Path:
    """Генерируем ответы учителя батчами и сохраняем их в answers.jsonl.

    Args:
        model: модель-учитель.
        tokenizer: общий токенизатор учителя и ученика.
        rows: примеры train с полями `id`, `text`, `gold`.
        output_path: пустой каталог для ответов.

    Returns:
        Каталог с answers.jsonl; в каждой строке `id`, `text`, `gold`,
        `teacher_answer`, `prompt_ids`, `answer_ids`.
    """
    output_path = ensure_empty_dir(output_path)
    records = []
    row_list = list(rows)

    for start in tqdm(range(0, len(row_list), GENERATE_BATCH_SIZE), desc="teacher train"):
        batch = row_list[start:start+GENERATE_BATCH_SIZE]
        prompts = [encode_prompt(tokenizer, row["text"]) for row in batch]
        answers = generate_answer_ids(model, teacher_tokenizer, [encode_prompt(teacher_tokenizer, row["text"]) for row in batch])
        for row, prompt, answer in zip(batch, prompts, answers):
            text = teacher_tokenizer.decode(answer, skip_special_tokens=True)
            student_ids = tokenizer.encode(text, add_special_tokens=False) + [tokenizer.eos_token_id]
            records.append({**row, "prompt_ids": prompt, "answer_ids": student_ids, "teacher_answer": text})

    if not records:
        raise ValueError("Учитель не вернул ни одного ответа.")
    (output_path / "answers.jsonl").write_text(
        "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in records),
        encoding="utf-8",
    )
    print(f"Сохранено ответов: {len(records)} / {len(row_list)}")
    return output_path

In [6]:
distill_rows = train_data.shuffle(seed=SEED).select(
    range(min(DISTILL_SAMPLES, len(train_data)))
)
TEACHER_ANSWERS = collect_teacher_answers(
    teacher, tokenizer, distill_rows, ARTIFACTS / "teacher_answers"
)
del teacher
clean_memory()

teacher train: 100%|██████████| 128/128 [09:03<00:00,  4.25s/it]


Сохранено ответов: 1024 / 1024


### Реализация пайплайна обучения ученика (1 балл)

Полное дообучение Qwen3-8B на A100 80GB проходит на грани по памяти. Для решения можно включать gradient checkpointing, уменьшать batch или брать меньшую модель того же семейства.

In [7]:
class TeacherAnswerDataset(torch.utils.data.Dataset):
    """Примеры для обучения ученика на ответах учителя."""

    def __init__(self, path: str | Path) -> None:
        """

        Args:
            path: каталог с answers.jsonl из collect_teacher_answers.
        """
        self.rows = [] # Список примеров из файла

        self.rows = [json.loads(line) for line in (Path(path) / "answers.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
        self.rows = [row for row in self.rows if row["answer_ids"]]

    def __len__(self) -> int:
        """
            Число строк в answers.jsonl.
        """
        return len(self.rows)

    def __getitem__(self, index: int) -> dict[str, list[int]]:
        """
        Args:
            index: номер примера.

        Returns:
            Словарь с `input_ids` (промпт + ответ), `attention_mask` и `labels`,
            токены промпта маскируем значением -100.
        """
        row = self.rows[index]
        input_ids = row["prompt_ids"] + row["answer_ids"]
        labels = [-100] * len(row["prompt_ids"]) + row["answer_ids"]
        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels,
        }

def collate_teacher_answers(features: list[dict[str, list[int]]]) -> dict[str, torch.Tensor]:
    """Паддим примеры справа до общей длины батча.

    Args:
        features: примеры из TeacherAnswerDataset.

    Returns:
        Тензоры `input_ids`, `attention_mask` и `labels` формы [batch, length];
        паддинг в `labels` равен -100.
    """
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id
    width = max(len(feature["input_ids"]) for feature in features)
    batch = {"input_ids": [], "attention_mask": [], "labels": []}
    for feature in features:
        pad = width - len(feature["input_ids"])
        batch["input_ids"].append(feature["input_ids"] + [pad_id] * pad)
        batch["attention_mask"].append(feature["attention_mask"] + [0] * pad)
        batch["labels"].append(feature["labels"] + [-100] * pad)
    return {key: torch.tensor(value) for key, value in batch.items()}

In [8]:
from transformers import Trainer, TrainingArguments


train_answers = TeacherAnswerDataset(TEACHER_ANSWERS)
train_answers.rows = [row for row in train_answers.rows
                      if parse_answer(row["teacher_answer"], INTENTS, SLOT_TYPES) is not None]
if not train_answers.rows:
    raise ValueError("Нет валидных ответов учителя")
print("Valid teacher answers:", len(train_answers))

def train_student(name, lr, epochs, n):
    set_seed(SEED)
    student = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL_ID, torch_dtype=torch.bfloat16)
    student.config.use_cache = False
    subset = torch.utils.data.Subset(train_answers, range(min(n, len(train_answers))))
    args = TrainingArguments(output_dir=str(ARTIFACTS / name), learning_rate=lr,
        num_train_epochs=epochs, per_device_train_batch_size=1,
        gradient_accumulation_steps=8, gradient_checkpointing=True,
        bf16=True, optim="adamw_torch", logging_steps=10, save_strategy="no",
        report_to="none", seed=SEED, remove_unused_columns=False)
    sft = Trainer(model=student, args=args, train_dataset=subset, data_collator=collate_teacher_answers)
    train_output = sft.train()
    student.config.use_cache = True
    path = ARTIFACTS / name
    sft.save_model(str(path))
    tokenizer.save_pretrained(path)
    del sft, student
    clean_memory()
    metrics, preds = evaluate_checkpoint(path, dev_data)
    return save_eval(name, metrics, preds, learning_rate=lr, epochs=epochs,
                     train_samples=len(subset), train_loss=train_output.training_loss)


Valid teacher answers: 1002


In [9]:
distill_results = [train_student("sft_1", 2e-5, 2, 128)]


Step,Training Loss
10,0.363800


dev: 100%|██████████| 32/32 [00:43<00:00,  1.36s/it]


### Добейтесь максимального качества ученика. Цель — не более 10% отклонения от учителя (2 балла)
Пробуйте менять гиперпараметры, число сэмплов, модели ученика (при условии размера модели ученика не более 8B).

In [10]:
for name, lr, epochs, n in [("sft_2", 2e-5, 2, DISTILL_SAMPLES),
                            ("sft_3", 5e-5, 3, DISTILL_SAMPLES)]:
    distill_results.append(train_student(name, lr, epochs, n))
    pd.DataFrame(distill_results).to_csv(ARTIFACTS / "distillation_results.csv", index=False)
best = max(distill_results, key=lambda row: row["quality"])
shutil.copytree(ARTIFACTS / best["run"], ARTIFACTS / "student_sft")
student_metrics, student_predictions = evaluate_checkpoint(ARTIFACTS / "student_sft", dev_data)
save_eval("student_sft", student_metrics, student_predictions, selected_run=best["run"])
summary = {"baseline": baseline_metrics, "teacher": teacher_metrics, "student": student_metrics,
           "quantization": results, "distillation": distill_results,
           "selected_run": best["run"], "student_model": STUDENT_MODEL_ID,
           "teacher_model": TEACHER_MODEL_ID, "seed": SEED,
           "target_reached": student_metrics["quality"] >= 0.9 * teacher_metrics["quality"],
           "beats_baseline": student_metrics["quality"] > baseline_metrics["quality"]}
(ARTIFACTS / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
display(pd.DataFrame([baseline_metrics, teacher_metrics, student_metrics], index=["baseline", "teacher", "student"]))


Step,Training Loss
10,0.376500
20,0.198100
30,0.205100
40,0.156800
50,0.188700
60,0.164000
70,0.121500
80,0.102000
90,0.103900
100,0.114900


dev: 100%|██████████| 32/32 [00:34<00:00,  1.06s/it]


Step,Training Loss
10,0.467600
20,0.246800
30,0.242700
40,0.187400
50,0.216900
60,0.183300
70,0.125500
80,0.102400
90,0.094400
100,0.112300


dev: 100%|██████████| 32/32 [00:34<00:00,  1.09s/it]


,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate,n
baseline,0.084531,0.140625,0.028436,0.007812,0.421875,256
teacher,0.585603,0.789062,0.382143,0.320312,0.980469,256
student,0.557622,0.738281,0.376963,0.289062,0.992188,256


### Выводы
Заполняются по фактическим результатам запуска (summary.json).